# Trabalho Final - Parte 2 - Parcial

# Experimentos

- Sempre 8 processos
- Implementações com multiprocessamento
  - Uso do módulo **multiprocessing**
- Cada processo em um core (físico ou lógico)
  - Sem alocação de 2 processos/threads no mesmo core
  - **Definição de afinidade**: Cada thread em um core
- Setar governor para **performance**

# Aplicação: Multiplicação de Matrizes

In [2]:
"""
Safe Matrix Multiplication Script (multiprocessing + psutil)
------------------------------------------------------------
- MATRIX_SIZE is configurable (default 16000)
- NUM_PROCESSES = 8 processes in parallel
- Each process is pinned to a specific CPU using psutil
"""

import os
import sys
import time
import psutil
import numpy as np
import multiprocessing


# -----------------------------
# CONFIGURATION
MATRIX_SIZE = 8000    # Size of NxN matrices
NUM_PROCESSES = 8      # Number of worker processes


# -----------------------------
# Safety check
if MATRIX_SIZE % NUM_PROCESSES != 0:
    print(f"Error: MATRIX_SIZE ({MATRIX_SIZE}) is not divisible by NUM_PROCESSES ({NUM_PROCESSES}).")
    sys.exit(1)


# -----------------------------
def worker_multiply(start_row, end_row, A_slice, B, cpu_id, return_dict, index):
    """
    Worker process: multiplies a slice of A with full B.
    Each worker is pinned to a specific CPU.
    """
    pid = os.getpid()
    proc = psutil.Process(pid)
    proc.cpu_affinity([cpu_id])  # fixa afinidade no CPU escolhido
    print(f"Process {pid} handling rows {start_row}:{end_row}, pinned to CPU {cpu_id}")

    # Multiplica fatia
    result = A_slice @ B
    return_dict[index] = result


def parallel_matrix_multiply(A, B, num_processes=NUM_PROCESSES):
    """
    Multiplica A e B em paralelo usando num_processes processos.
    """
    n = A.shape[0]
    rows_per_proc = n // num_processes

    manager = multiprocessing.Manager()
    return_dict = manager.dict()
    processes = []

    # CPUs disponíveis
    available_cpus = psutil.cpu_count(logical=True)
    cpu_ids = list(range(min(num_processes, available_cpus)))

    for i in range(num_processes):
        start_row = i * rows_per_proc
        end_row = (i + 1) * rows_per_proc
        A_slice = A[start_row:end_row, :]

        p = multiprocessing.Process(
            target=worker_multiply,
            args=(start_row, end_row, A_slice, B, cpu_ids[i % available_cpus], return_dict, i)
        )
        processes.append(p)
        p.start()

    for p in processes:
        p.join()

    # Reconstrói na ordem certa
    result_slices = [return_dict[i] for i in range(num_processes)]
    return np.vstack(result_slices)


# -----------------------------
def run_benchmark(matrix_size):
    print(f"\n=== Benchmark: {matrix_size}x{matrix_size} matrix multiplication ===")
    A = np.random.random((matrix_size, matrix_size))
    B = np.random.random((matrix_size, matrix_size))

    start = time.perf_counter()
    C = parallel_matrix_multiply(A, B, NUM_PROCESSES)
    end = time.perf_counter()

    print("Matrix multiplication completed")
    print(f"Result shape: {C.shape}")
    print(f"Elapsed time: {end - start:.2f} seconds")


# -----------------------------
if __name__ == "__main__":
    run_benchmark(MATRIX_SIZE)



=== Benchmark: 8000x8000 matrix multiplication ===
Process 8933 handling rows 0:1000, pinned to CPU 0
Process 8937 handling rows 1000:2000, pinned to CPU 1
Process 8943 handling rows 2000:3000, pinned to CPU 2
Process 8949 handling rows 3000:4000, pinned to CPU 3
Process 8955 handling rows 4000:5000, pinned to CPU 4
Process 8961 handling rows 5000:6000, pinned to CPU 5Process 8967 handling rows 6000:7000, pinned to CPU 6

Process 8973 handling rows 7000:8000, pinned to CPU 7
Matrix multiplication completed
Result shape: (8000, 8000)
Elapsed time: 55.77 seconds
